# OD Reading vs. Hours Since Experiment

This notebook lets you load a CSV file and plot `od_reading` vs `hours_since_experiment_created`, with one trace per pioreactor unit.

It also includes optional normalization cells so you can compare trajectories across units after baseline scaling.

In [40]:
import os, sys
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — faster, prevents memory leaks in Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from scipy.signal import medfilt
from scipy.stats import linregress
import numpy as np

## 1. Select File

In [41]:
IN_COLAB = 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG') is not None

if IN_COLAB:
    from google.colab import files
    from pathlib import Path
    print("Running in Google Colab — please upload your CSV file:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    CSVPATH = Path(fname)
else:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    CSVPATH = filedialog.askopenfilename(
        title='Select CSV file',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()
    if not CSVPATH:
        raise FileNotFoundError('No file was selected.')
    CSVPATH = Path(CSVPATH).expanduser().resolve()

print(f'Selected file: {CSVPATH}')

Running in Google Colab — please upload your CSV file:


Saving od_readings-JUNE262026-all_units-20260424233439.csv to od_readings-JUNE262026-all_units-20260424233439 (4).csv
Selected file: od_readings-JUNE262026-all_units-20260424233439 (4).csv


## 2. Load Data

In [42]:
USECOLS = ['pioreactor_unit', 'hours_since_experiment_created', 'od_reading', 'channel']
df = pd.read_csv(
    CSVPATH,
    usecols=lambda c: c in USECOLS,
    dtype={'pioreactor_unit': 'category', 'od_reading': 'float32', 'channel': 'float32'},
)

df = df.rename(columns={
    'hours_since_experiment_created': 'hourssince',
    'pioreactor_unit': 'pioreactor',
})
df['pioreactor'] = df['pioreactor'].astype('category')

print(f'Rows: {len(df):,}  Columns: {list(df.columns)}')
display(df.head(3))

OUTPUTDIR = CSVPATH.parent
print(f'Output directory: {OUTPUTDIR}')

Rows: 402,919  Columns: ['pioreactor', 'od_reading', 'channel', 'hourssince']


,pioreactor,od_reading,channel,hourssince
0,pioreactor01,0.0,2.0,-0.941111
1,pioreactor01,0.0,2.0,-0.939722
2,pioreactor01,0.0,2.0,-0.938333


Output directory: .


# Filter by Channel

In [43]:
CHANNEL = None  # Set to e.g. 2 to filter to a single channel
CHANNEL = None  # set to e.g. 2 to filter

if CHANNEL is not None:
    dfplot = df[df['channel'] == CHANNEL].copy()
    print(f'Filtered to channel {CHANNEL}: {len(dfplot)} rows')
else:
    dfplot = df.copy()
    print(f'Using all channels: {len(dfplot)} rows')

Using all channels: 402919 rows


In [44]:
print("Raw NaN in od_reading:", df['od_reading'].isna().sum())
print("By unit:")
print(df.groupby('pioreactor', observed=True)['od_reading'].apply(lambda x: x.isna().sum()))

Raw NaN in od_reading: 0
By unit:
pioreactor
pioreactor01    0
pioreactor02    0
pioreactor03    0
pioreactor04    0
pioreactor05    0
pioreactor06    0
Name: od_reading, dtype: int64


In [45]:
df = df.dropna(subset=['od_reading'])
print(f'After dropping NaN od_reading: {len(df):,} rows')

After dropping NaN od_reading: 402,919 rows


## 3. Optional Downsample for Fast Plotting

The dataset can be large. If plotting is slow, uncomment the downsample cell below
to keep every Nth point per unit (a rolling median is applied first to reduce noise).

In [46]:
#CHANNEL = None  # Set to e.g. 2 to filter to a single channel

#dfplot = df[df['channel'] == CHANNEL].copy() if CHANNEL is not None else df.copy()
#print(f'Rows for plotting: {len(dfplot):,}')

#units   = sorted(dfplot['pioreactor'].unique())
#palette = sns.color_palette('tab10', len(units))
#ncols   = 3
#nrows   = -(-len(units) // ncols)

## 4. Create Normalized OD Values

This section creates normalized versions of `od_reading` for each pioreactor.

Included normalizations:
- `od_norm_t0`: divides by the first OD value for each pioreactor.
- `od_delta_t0`: subtracts the first OD value for each pioreactor.
- `od_minmax`: rescales each pioreactor trace between 0 and 1.

In [47]:
dfplot = dfplot.sort_values(['pioreactor', 'hourssince']).copy()

first_od  = dfplot.groupby('pioreactor')['od_reading'].transform('first')
min_od    = dfplot.groupby('pioreactor')['od_reading'].transform('min')
max_od    = dfplot.groupby('pioreactor')['od_reading'].transform('max')
range_od  = (max_od - min_od).replace(0, pd.NA)

dfplot['od_norm_t0']  = dfplot['od_reading'] / first_od
dfplot['od_delta_t0'] = dfplot['od_reading'] - first_od
dfplot['od_minmax']   = (dfplot['od_reading'] - min_od) / range_od

cols_to_show = ['pioreactor', 'hourssince', 'od_reading', 'od_norm_t0', 'od_delta_t0', 'od_minmax']
dfplot[cols_to_show].head()

/tmp/ipykernel_17890/4043722207.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_od  = dfplot.groupby('pioreactor')['od_reading'].transform('first')
/tmp/ipykernel_17890/4043722207.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  min_od    = dfplot.groupby('pioreactor')['od_reading'].transform('min')
/tmp/ipykernel_17890/4043722207.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_od    = dfplot.groupby

,pioreactor,hourssince,od_reading,od_norm_t0,od_delta_t0,od_minmax
0,pioreactor01,-0.941111,0.0,NaN,0.0,0.0
1,pioreactor01,-0.939722,0.0,NaN,0.0,0.0
2,pioreactor01,-0.938333,0.0,NaN,0.0,0.0
8,pioreactor01,-0.936944,0.0,NaN,0.0,0.0
14,pioreactor01,-0.935556,0.0,NaN,0.0,0.0


## 6. Plot Normalised OD (t₀)

In [48]:
fig, ax = plt.subplots(figsize=(12, 5))
for unit, grp in dfplot.groupby('pioreactor'):
    ax.plot(grp['hourssince'], grp['od_norm_t0'], label=unit, alpha=0.8)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('OD / OD(t=0)')
ax.set_title('Normalized OD (relative to first reading)')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_norm_t0.png', dpi=150)
plt.close(fig)
print('Saved od_norm_t0.png')

/tmp/ipykernel_17890/543123599.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for unit, grp in dfplot.groupby('pioreactor'):


Saved od_norm_t0.png


## 7. Plot: Raw OD — All Units Overlays

In [49]:
fig, ax = plt.subplots(figsize=(14, 5))
for unit, grp in dfplot.groupby('pioreactor'):
    ax.plot(grp['hourssince'], grp['od_reading'], label=unit, alpha=0.7, linewidth=0.8)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title('od_reading vs. Time — All Pioreactor Units')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_all_units.png', dpi=150)
plt.close(fig)
print('Saved od_reading_vs_hours_all_units.png')

/tmp/ipykernel_17890/3209536012.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for unit, grp in dfplot.groupby('pioreactor'):


Saved od_reading_vs_hours_all_units.png


## 8. Plot Raw OD — Individual Unit Subplots

In [50]:
units   = sorted(dfplot['pioreactor'].unique())
ncols   = 3
nrows   = -(-len(units) // ncols)
palette = sns.color_palette('tab10', len(units))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharey=False)
axes = axes.flatten()

# Pre-group ONCE — reused by cells 10 and 11
groups = {u: g for u, g in dfplot.groupby('pioreactor')}

for i, (unit, color) in enumerate(zip(units, palette)):
    grp = groups[unit]
    axes[i].plot(grp['hourssince'], grp['od_reading'], color=color, linewidth=0.8)
    axes[i].set_title(unit, fontweight='bold')
    axes[i].set_xlabel('Hours since start')
    axes[i].set_ylabel('od_reading')
    axes[i].grid(True, linestyle='--', alpha=0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('od_reading vs. Time — Individual Units', fontweight='bold', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_subplots.png', dpi=150)
plt.close(fig)
print('Saved od_reading_vs_hours_subplots.png')

/tmp/ipykernel_17890/1706675418.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = {u: g for u, g in dfplot.groupby('pioreactor')}


Saved od_reading_vs_hours_subplots.png


## 9 Plot Smoothed OD — Rolling Median Overlay

In [51]:
SMOOTH_WIN = 60

fig, ax = plt.subplots(figsize=(14, 5))
palette = sns.color_palette('tab10', len(units))

for unit, color in zip(units, palette):
    grp      = groups[unit].sort_values('hourssince')
    smoothed = grp['od_reading'].rolling(SMOOTH_WIN, center=True, min_periods=1).median()
    ax.plot(grp['hourssince'], grp['od_reading'], color=color, alpha=0.2, linewidth=0.5)
    ax.plot(grp['hourssince'], smoothed,          color=color, linewidth=1.5, label=unit)

ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title(f'od_reading vs. Time — Smoothed (rolling median, window={SMOOTH_WIN})')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_smoothed.png', dpi=150)
plt.close(fig)
print('Saved od_reading_vs_hours_smoothed.png')

Saved od_reading_vs_hours_smoothed.png


## 10. Find Doubling Time — Exponential Fit

Finds the best local exponential growth window per unit using a sliding window on log(OD),
then reports growth rate *r* (h⁻¹), doubling time *Td* = ln(2)/r, and R².

In [52]:
WIN_H      = 3.0   # sliding window width in hours
MIN_OD     = 0.05  # ignore readings below this (noise floor)
MEDFILT_K  = 51    # robust smoothing kernel (must be odd)
T_STEP     = None  # None = use every time point; set e.g. 0.1 to coarsen the sweep

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), sharey=False)
axes = axes.flatten()
palette = sns.color_palette('tab10', len(units))

summary_rows = []

for i, (unit, color) in enumerate(zip(units, palette)):
    grp = groups[unit].sort_values('hourssince').copy()
    grp = grp[grp['od_reading'] > MIN_OD].reset_index(drop=True)
    if len(grp) < 20:
        continue

    t  = grp['hourssince'].values.astype(float)
    od = grp['od_reading'].values.astype(float)

    k      = min(MEDFILT_K, len(od) if len(od) % 2 == 1 else len(od) - 1)
    od_s   = medfilt(od, kernel_size=k)
    log_od = np.log(np.clip(od_s, 1e-9, None))

    # O(N log N) sweep — searchsorted finds window edges without scanning full array
    starts = np.arange(t[0], t[-1] - WIN_H, T_STEP) if T_STEP else t
    best_r2, best_r, best_td, best_mask = -np.inf, np.nan, np.nan, None
    best_intercept = np.nan

    for t0 in starts:
        lo = np.searchsorted(t, t0,         side='left')
        hi = np.searchsorted(t, t0 + WIN_H, side='right')
        if hi - lo < 5:
            continue
        tw = t[lo:hi];  lw = log_od[lo:hi]
        slope, intercept, r, _, _ = linregress(tw, lw)
        if r**2 > best_r2 and slope > 0:
            best_r2        = r**2
            best_r         = slope
            best_td        = np.log(2) / slope
            best_mask      = slice(lo, hi)
            best_intercept = intercept

    ax = axes[i]
    ax.plot(t, od,   color=color, alpha=0.2, linewidth=0.5)
    ax.plot(t, od_s, color=color, linewidth=1.5)

    if best_mask is not None:
        t_win = t[best_mask]
        ax.fill_betweenx([0, od.max() * 1.1], t_win[0], t_win[-1],
                          color='gray', alpha=0.15)
        ax.plot(t_win, np.exp(best_intercept + best_r * t_win),
                'k--', linewidth=1.5, label='Exp fit')
        ax.text(0.02, 0.97,
                f'r={best_r:.3f} h\u207b\u00b9\nTd={best_td:.2f} h\nR\u00b2={best_r2:.3f}',
                transform=ax.transAxes, va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
        summary_rows.append(dict(unit=unit, r=best_r, Td_h=best_td, R2=best_r2,
                                  win_start=t_win[0], win_end=t_win[-1]))

    ax.set_title(unit, fontweight='bold')
    ax.set_xlabel('Hours since experiment created')
    ax.set_ylabel('od_reading')
    ax.grid(True, linestyle='--', alpha=0.5)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('od_reading with robust smoothing and best local exponential fit',
             fontweight='bold', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_robust_expfit_subplots.png', dpi=150)
plt.close(fig)
print('Saved od_reading_robust_expfit_subplots.png')

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index('unit'))

Saved od_reading_robust_expfit_subplots.png


,r,Td_h,R2,win_start,win_end
unit,,,,,
pioreactor01,0.335558,2.065652,0.983497,6.576944,9.575556
pioreactor02,0.329703,2.102339,0.985475,6.854444,9.853056
pioreactor03,0.322289,2.150704,0.980674,7.314167,10.312778
pioreactor04,0.424100,1.634394,0.986138,6.480833,9.479444
pioreactor05,0.717246,0.966400,0.991400,4.669722,7.669722
pioreactor06,0.416467,1.664349,0.984367,6.311389,9.310000


In [53]:
from google.colab import files
import glob

for png in glob.glob(str(OUTPUTDIR / '*.png')):
    files.download(png)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>